Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.functions import trim,col

Read Bronze Table

In [0]:
df=spark.table("workspace.bronze.loc_a101")

Silver Transformations

Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType,StringType):
        df=df.withColumn(field.name,trim(col(field.name)))


Customer ID Cleanup

In [0]:
df=df.withColumn("cid",F.regexp_replace("cid","-",""))

Country Normalization

In [0]:
df = df.withColumn(
    "cntry",
    F.when(col("cntry") == "DE", "Germany")
        .when(col("cntry").isin("US", "USA"), "United States")
        .when(
            (col("cntry") == "") | col("cntry").isNull(),
            "n/a"
        )
        .otherwise(col("cntry"))
)

Renaming Columns

In [0]:
RENAME_MAP={
    "cntry":"country",
    "cid":"customer_number"
}
for old,new in RENAME_MAP.items():
    df=df.withColumnRenamed(old,new)

Sanity check of dataframe 

In [0]:
df.limit(10).display()

Write in Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_custLocation")